# v8 environment-option results (Oct 31 - Nov 1, 2026)

Views the eg15 runs launched on Oct 31: the 512M one-factor wave, the 2B wave and the overnight queue.
Each run directory holds `curve.json` (validation every checkpoint), `summary.json`, `protocol.json`
and `diag.json` (stockpiling diagnostic on the final checkpoint). All evaluations use the same 128
validation worlds (23,000,001-23,000,128); scores are achievements out of 32.

Run with pandas and matplotlib, e.g. from the repo root:

    uv run --no-project --with pandas,matplotlib,jupyterlab jupyter lab notebooks/v8_env_results_2026_10_31.ipynb

To refresh data from eg15 (curves, summaries, protocols, diagnostics only):

    rsync -a -e "ssh -J decidueye" --include='*/' --include='curve.json' --include='summary.json' \
      --include='protocol.json' --include='diag.json' --exclude='*' \
      mvohl@cv350-rck-g03-c01-08.rck.dcgpu:Projects/RL/experiments/<study>/ experiments/eg15/<study>/

In [ ]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

DATA = Path("../experiments/eg15") if Path("../experiments/eg15").exists() else Path("experiments/eg15")
STUDIES = {  # label -> directory under DATA; add new studies here
    "512M wave": "eg15-envopt-v7-20261027",
    "2B wave": "eg15-envopt2b-v8-20261101",
    "overnight": "eg15-overnight-v8-20261101",
}
# Library build directory (or overnight job-name prefix) -> environment variant.
VARIANTS = {
    "build-v8": "v8 default (= v7)", "build-v8-ore200": "ore 2x", "build-v8-ore300": "ore 3x",
    "build-v8-tree200": "trees 2x", "build-v8-ore300-tree200": "ore 3x + trees 2x",
    "build-v8-view21": "view 21x15", "build-v8-real": "realistic (ore 3x + trees 2x + view 21x15)",
    "build-v8-w256": "world 256x96", "build-v8-o3t2-inv": "ore 3x + trees 2x + log inventory",
    "build-v8-real-inv": "realistic + log inventory",
    # overnight job-name prefixes
    "o3t2": "ore 3x + trees 2x", "o3t2inv": "ore 3x + trees 2x + log inventory",
    "real": "realistic (ore 3x + trees 2x + view 21x15)", "realinv": "realistic + log inventory",
    "w256": "world 256x96",
}
HARD = ["craft anvil", "craft metal sword", "craft iron pickaxe", "craft gold pickaxe",
        "craft iron chainmail", "craft wood breastplate", "return to surface"]

In [ ]:
def variant_of(group_dir: str) -> str:
    head, _, tail = group_dir.rpartition("-")      # strip "-g3" (GPU) or "-2001" (seed) suffixes
    name = head if tail.lstrip("g").isdigit() else group_dir
    if name in VARIANTS:
        return VARIANTS[name]
    prefix = name.split("_")[0]          # overnight job names, e.g. o3t2inv_fn_unimix-2001
    return VARIANTS.get(prefix, name)

def arm_of(cfg: dict, episode_steps: int) -> str:
    parts = [f"pot {cfg['potential_coef']:g}" + (f" {cfg['potential_mode']}" if cfg["potential_mode"] != "mean" else "")]
    if cfg.get("novelty_coef"): parts.append(f"novelty {cfg['novelty_coef']:g}")
    if cfg.get("unimix"): parts.append(f"unimix {cfg['unimix']:g}")
    if cfg.get("memory"): parts.append(f"{cfg['memory']} {cfg.get('memory_size') or 256}")
    if cfg.get("hidden"): parts.append(f"hidden {cfg['hidden']}")
    if cfg.get("lr") != 3e-4: parts.append(f"lr {cfg['lr']:g}")
    if episode_steps != 3000: parts.append(f"{episode_steps} steps")
    return ", ".join(parts)

rows, curves = [], []
for study, sub in STUDIES.items():
    for run in sorted((DATA / sub).glob("*/runs/*")):
        if not (run / "curve.json").exists():
            continue
        curve = json.loads((run / "curve.json").read_text())
        proto = json.loads((run / "protocol.json").read_text())
        ev = curve[-1]["evaluation"]
        diag = json.loads((run / "diag.json").read_text()) if (run / "diag.json").exists() else None
        key = dict(study=study, variant=variant_of(run.parent.parent.name),
                   arm=arm_of(proto["config"], proto.get("episode_steps", 3000)),
                   budget=f"{proto['actual_steps'] / 2**30:.0f}B" if proto["actual_steps"] >= 2**30
                          else f"{proto['actual_steps'] / 2**20:.0f}M",
                   seed=int(run.name.rsplit("-", 1)[1]))
        row = dict(key, steps_B=curve[-1]["steps"] / 2**30, finished=(run / "summary.json").exists(),
                   score=ev["mean_achievements"], best=max(c["evaluation"]["mean_achievements"] for c in curve),
                   deaths=ev["death_rate"], idle_steps=ev["mean_steps_after_last_achievement"],
                   **{h: ev["hits"].get(h, 0) for h in HARD})
        if diag:
            peak = diag["inventory_peak"]
            row.update(iron_bars_med=peak["iron bar"]["median"], iron_bars_max=peak["iron bar"]["max"],
                       wood_med=peak["wood"]["median"], wood_max=peak["wood"]["max"],
                       breastplate_legal_eps=diag["actions"]["craft wood breastplate"]["episodes_with_opportunity"],
                       iron_pick_legal_eps=diag["actions"]["craft iron pickaxe"]["episodes_with_opportunity"])
        rows.append(row)
        curves += [dict(key, steps_B=c["steps"] / 2**30, score=c["evaluation"]["mean_achievements"]) for c in curve]

runs = pd.DataFrame(rows)
curves = pd.DataFrame(curves)
print(f"{len(runs)} runs, {runs.finished.sum()} finished")
runs.sort_values(["study", "variant", "arm", "seed"]).reset_index(drop=True)

## Summary per study, variant and arm

Means over seeds. Hard-tier columns are validation worlds out of 128; `*_med` are median per-episode
peaks of items held at once (stockpiling); `*_legal_eps` count episodes where the craft was ever possible.

In [ ]:
cols = ["score", "best", "deaths", *HARD, "iron_bars_med", "wood_med", "breastplate_legal_eps", "iron_pick_legal_eps"]
summary = (runs.groupby(["study", "variant", "arm", "budget"])
               .agg(seeds=("seed", "count"), **{c: (c, "mean") for c in cols if c in runs})
               .round(2).sort_values(["study", "score"], ascending=[True, False]))
summary

## Learning curves (mean over seeds)

In [ ]:
def plot(study, ax):
    sub = curves[curves.study == study]
    for (variant, arm, budget), g in sub.groupby(["variant", "arm", "budget"]):
        m = g.groupby("steps_B").score.mean()
        ax.plot(m.index, m.values, marker=".", label=f"{variant} | {arm} ({budget})")
    ax.set(title=study, xlabel="env steps (billions)", ylabel="validation score /32")
    ax.grid(alpha=.3)
    ax.legend(fontsize=7, loc="lower right")

fig, axes = plt.subplots(1, len(STUDIES), figsize=(7 * len(STUDIES), 6))
for study, ax in zip(STUDIES, axes):
    plot(study, ax)
plt.tight_layout()

## Hard achievements of the best arms (worlds out of 128, mean over seeds)

In [ ]:
top = summary.reset_index().sort_values("score", ascending=False).head(10)
top = top.assign(label=top.variant + " | " + top.arm + " (" + top.study + ")").set_index("label")
top[HARD].plot.barh(figsize=(10, 7), width=.8)
plt.xlabel("worlds out of 128")
plt.gca().invert_yaxis()
plt.tight_layout()

## Per-seed detail for one variant
Change `VARIANT` to inspect seed variance (e.g. return to surface differs a lot between seeds).

In [ ]:
VARIANT = "ore 3x + trees 2x + log inventory"
runs[runs.variant == VARIANT].sort_values(["arm", "seed"])[["study", "arm", "seed", "steps_B", "score", *HARD, "iron_bars_med", "wood_med"]]